# Konu 11 — RAG: belgeye dayalı cevap

Hücreleri **yukarıdan aşağı, sırayla** çalıştır: hücreye tıkla, **Shift + Enter**.
Her hücre bir öncekinin oluşturduğu değişkeni kullanır; atlarsan hata alırsın.

Konu 5'te gördük: model, bilmediği bir şey sorulunca çoğu zaman "bilmiyorum" demez; olası görünen bir cevap uydurur.
Bugünkü örnek belge, sessiz çalışma kafemizin marka kılavuzu: `veri/kilavuz.txt`. Kafenin adı **Sayfa Kahve**.
Modele soru sormadan önce kılavuzdan soruya en ilgili paragrafları bulacak ve yönergeye koyacağız; model cevabı
kılavuza bakarak verecek. Bu yöntemin adı **RAG**.

**Anahtar ve kota:** Isınma anahtarsız. Adım 1'den sonrası `anahtar.txt` ister. Defterin tamamı (Bonus hariç)
5 vektör isteği, 4 metin isteği ve 1 görsel isteği gönderir. Kılavuzun 14 paragrafı **tek istekte** vektöre çevrilir
ve dosyaya kaydedilir; defteri ikinci kez çalıştırdığında bu istek gitmez. Hepsi günlük ücretsiz kotanın küçük bir kısmı.

## Isınma — Kitaba bakarak cevap, kâğıt üstünde

Önce internetsiz, Konu 4'teki gibi elle verilmiş vektörlerle. Dört kısa paragraflık küçük bir kılavuz kuruyoruz.
Her paragrafa üç sayı verdik; her sayı, paragrafın bir konudan ne kadar söz ettiğini söylüyor (0 hiç, 1 tamamen):

- **renk** · **yazı tipi** · **açılış saati**

Sözlüğün anahtarı paragrafın kendisi, değeri onun üç sayısı (Konu 4'teki `harita` gibi).

In [ ]:
ornek_kilavuz = {
    "Ana renkler mürekkep mavisi ve kâğıt beyazıdır.": [0.9, 0.1, 0.0],
    "Başlıklarda serifli bir yazı tipi kullanılır.": [0.1, 0.9, 0.0],
    "Kafe hafta içi 08.00–23.00 arası açıktır.": [0.0, 0.0, 1.0],
    "Menü kartları beyaz zemine mavi mürekkeple, serifli başlıklarla basılır.": [0.6, 0.6, 0.0],
}
print(len(ornek_kilavuz))

Konu 4'teki `benzerlik` fonksiyonu: iki vektör aynı yöne bakıyorsa 1'e yakın, ilgisizse 0'a yakın bir sayı verir.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def benzerlik(a, b):
    sonuc = cosine_similarity([a], [b])
    satir = sonuc[0]
    return satir[0]

Soru: "Afişte hangi renkleri kullanayım?" Bu soru yalnızca renkten söz ediyor; üç sayısı `[1.0, 0.0, 0.0]`.
Konu 4'ün Adım 6'sındaki döngüyle her paragrafa bir puan veriyoruz: `skorlar` sözlüğü.

In [ ]:
soru = "Afişte hangi renkleri kullanayım?"
soru_vektoru = [1.0, 0.0, 0.0]

skorlar = {}
for paragraf in ornek_kilavuz:
    vektor = ornek_kilavuz[paragraf]
    skorlar[paragraf] = benzerlik(soru_vektoru, vektor)

print(len(skorlar))

**Tahmin et, sohbete yaz:** en yüksek puanı alan iki paragraf hangileri? Konu 4'teki gibi `Counter` en büyükleri sıralar.

In [ ]:
from collections import Counter

sirali = Counter(skorlar)
for paragraf, skor in sirali.most_common(2):
    print(round(skor, 2), paragraf)

İlk sırada ana renkler var (0.99). İkinci sırada menü kartları (0.71): içinde "beyaz" ve "mavi" de geçiyor.
Açılış saatinin puanı 0; renkle hiçbir ilgisi yok.

Şimdi bulduğumuz iki paragrafı, modele gidecek metnin, yani **yönergenin** içine koyuyoruz. `bilgiler` boş bir metinle
başlıyor; her turda bir paragraf ve bir yeni satır (`"\n"`) ekleniyor.

In [ ]:
bilgiler = ""
for paragraf, skor in sirali.most_common(2):
    bilgiler = bilgiler + paragraf + "\n"

yonerge = "Yalnızca şu bilgilere dayanarak cevap ver:\n" + bilgiler + "Soru: " + soru
print(yonerge)

RAG'in üç adımının üçü de burada, kâğıt üstünde:

1. **Parçala ve vektöre çevir:** her paragrafa sayılar (biz elle verdik).
2. **Bul:** sorunun vektörüne en yakın paragraflar (`skorlar`, `most_common`).
3. **Yönergeye koy ve sor:** paragraflar + soru. (Bu yönergeyi henüz kimseye göndermedik.)

Gerçekte sayıları model verecek (Konu 4'teki gibi yüzlerce sayı), yönergeyi de metin modeli okuyacak.
Kılavuz da dört değil, on dört paragraf.

## Adım 1 — Model Sayfa Kahve'yi tanıyor mu?

`rag_araclari.py` bu klasördeki yardımcı dosya; anahtarı `../anahtar.txt`'den kendisi okur. İçinde dört hazır fonksiyon var:

- `modele_sor(soru, sicaklik)`: Konu 5'te yazdığın fonksiyon. Tek farkı: istek başarısız olursa çökmez, `None` verir
  (Konu 10'daki gibi).
- `vektor_al(metin)`: Konu 4'te yazdığın fonksiyon; bir metnin gömme vektörünü verir.
- `vektorleri_al(metinler)`: aynı işin liste hâli (Adım 3).
- `ciz(yonerge, tohum, yol)`: Konu 8'deki fonksiyon; görseli verdiğin yola kaydeder (Adım 7).

In [ ]:
from rag_araclari import modele_sor, vektor_al, vektorleri_al, ciz

Önce kılavuz olmadan soralım. Sıcaklık 0.2, yani düşük (Konu 5): yaratıcılık değil, bilgi istiyoruz.

**Tahmin et, sohbete yaz:** model Sayfa Kahve'yi tanıyor mu? "Bilmiyorum" mu der, yoksa renk mi önerir?

In [ ]:
soru = "Sayfa Kahve'nin afişinde hangi renkleri kullanmalıyım?"

kilavuzsuz = modele_sor(soru, 0.2)
print(kilavuzsuz)

Senin çıktına bak:

- Model Sayfa Kahve'yi tanımadığını söyledi mi?
- Renk önerdiyse bu renkleri nereden buldu? Herhangi bir kafe için de aynılarını önermez miydi?
- Cevap kendinden emin mi, yoksa "genellikle", "olabilir" gibi çekinceler var mı?

Sayfa Kahve uydurma bir marka; kılavuzu modelin eğitim verisinde yok. Gerçek bir marka olsaydı da müşterinin geçen hafta
yazdığı kılavuz orada olmazdı. Model bilmediği yerde boşluğu en olası görünen metinle doldurur (Konu 5: uydurma).
Bu cevabı Adım 5'te kılavuzlu cevapla karşılaştıracağız.

## Adım 2 — Kılavuzu parçala

`veri/kilavuz.txt`'yi VS Code'da aç: Sayfa Kahve'nin marka kılavuzu. Her satır bir paragraf, her paragraf tek bir konu:
renkler, yazı tipi, logo, ses tonu...

RAG'in ilk işi belgeyi **parçalara** bölmek; bizim kılavuzda bir parça bir satır. Konu 1'deki gibi dosyayı satır satır
okuyoruz: `.strip()` satır sonundaki görünmez `\n`'yi siler, boş satırlar listeye girmez.

In [ ]:
paragraflar = []
with open("veri/kilavuz.txt", encoding="utf-8") as dosya:
    for satir in dosya:
        paragraf = satir.strip()
        if paragraf != "":
            paragraflar.append(paragraf)

print(len(paragraflar))
print(paragraflar[2])

Üçüncü paragraf (sıra numarası `2`: Python saymaya 0'dan başlar) kırmızıyı ve turuncuyu yasaklıyor.
Adım 1'deki cevapta bu renklerden biri var mıydı?

## Adım 3 — Paragrafları vektöre çevir ve sakla

Konu 4'te 24 kelime için 24 ayrı istek göndermiştik. Model tek istekte bir liste de alabiliyor:
`vektorleri_al(paragraflar)` 14 paragrafı **tek istekte** gönderir ve aynı sırayla 14 vektör verir.

Kılavuz değişmedikçe vektörleri de değişmez. Konu 10'un kuralı: **ara çıktıyı sakla.** Vektörleri bir dosyaya yazacağız:
`ciktilar/kilavuz_vektorleri.json`.

In [ ]:
import json
from pathlib import Path

klasor = Path("ciktilar")
klasor.mkdir(exist_ok=True)
vektor_dosyasi = klasor / "kilavuz_vektorleri.json"

Aşağıdaki hücre, Konu 10'daki slogan halkasının kalıbıyla çalışıyor. Önce sorar: **zaten var mı?**

- Dosya varsa: okur; modele hiç istek gitmez.
- Yoksa: vektörleri ister, her paragrafı kendi vektörüyle eşleştirir, sözlüğü dosyaya yazar.

Eşleştirme: birinci paragrafa birinci vektör, ikinciye ikinci... `sira` sayacı (Konu 1) hangi vektörde olduğumuzu tutar.
Sözlüğün anahtarı paragrafın kendisi, değeri onun vektörü (Konu 4'teki `vektorler_sozlugu` gibi).

In [ ]:
if vektor_dosyasi.exists():
    with open(vektor_dosyasi, encoding="utf-8") as dosya:
        kilavuz_vektorleri = json.load(dosya)
    print("Dosyadan okundu, istek gitmedi")
else:
    vektorler = vektorleri_al(paragraflar)
    kilavuz_vektorleri = {}
    sira = 0
    for paragraf in paragraflar:
        kilavuz_vektorleri[paragraf] = vektorler[sira]
        sira = sira + 1
    with open(vektor_dosyasi, "w", encoding="utf-8") as dosya:
        json.dump(kilavuz_vektorleri, dosya, ensure_ascii=False)
    print("Vektörler alındı ve kaydedildi")

print(len(kilavuz_vektorleri))

**Tahmin et:** bu hücreyi bir kez daha çalıştırırsan ne yazar? Kaç istek gider? Çalıştır ve bak.

`ciktilar/kilavuz_vektorleri.json`'u VS Code'da açarsan her paragrafın yanında yüzlerce sayı görürsün: kılavuzun modelin
gözündeki hâli.

**Dikkat:** kılavuza bir kural ekler ya da bir paragrafı değiştirirsen bu dosyayı sil, sonra Adım 2'den yeniden çalıştır.
Yoksa arama eski kılavuzda yapılır ve yeni kural hiç bulunmaz.

## Adım 4 — Soruya en ilgili paragraflar

Sorunun vektörü (Adım 1'deki soru):

In [ ]:
soru_vektoru = vektor_al(soru)
print(len(soru_vektoru))

Paragrafların vektörleriyle aynı uzunlukta; ancak böyle karşılaştırılabilirler.

**Tahmin et, sohbete yaz:** kılavuzda renkten söz eden kaç paragraf var? (Kılavuza bak.) En yüksek puanı alan üç paragraf
hangileri olur? Kırmızı–turuncu yasağı ilk üçe girer mi?

Bu adımı **sen yaz**. Plan ısınmadakiyle ve Konu 4'ün Adım 6'sıyla aynı:

1. `skorlar` adında boş bir sözlük kur.
2. `kilavuz_vektorleri`'ndeki her paragraf için sorunun vektörüyle paragrafın vektörünün benzerliğini hesapla,
   `skorlar[paragraf]` olarak sakla.
3. `Counter` ile sırala; en büyük üçünün puanını ve paragrafını yazdır.

`benzerlik` ve `Counter` ısınmada tanımlandı; ısınmayı atladıysan `NameError` alırsın. Aşağıdaki boş hücreye yaz;
altındaki çözüme bakmadan önce kendin dene.

Bir çözüm:

In [ ]:
skorlar = {}
for paragraf in kilavuz_vektorleri:
    vektor = kilavuz_vektorleri[paragraf]
    skorlar[paragraf] = benzerlik(soru_vektoru, vektor)

sirali = Counter(skorlar)
for paragraf, skor in sirali.most_common(3):
    print(round(skor, 2), paragraf)

Tahminin tuttu mu? Renk bilgisi kılavuzda tek bir yerde değil, altı paragrafa dağılmış: ana renkler, yasak renkler,
fotoğraf ışığı, çizim tarzı, mevsim vurgusu, menü kartı. Biz yalnızca ilk üçü alıyoruz; ilk üçe girmeyen paragraf modele
**hiç gitmeyecek**.

- Dışarıda kalan renk paragrafları hangileri?
- Bir afiş için dışarıda kalmaması gereken hangisiydi?

## Adım 5 — Yönerge şablonu: kılavuza bakarak sor

RAG'in üçüncü adımı: bulunan paragrafları yönergeye koyup soruyu sormak. Önce ilk üç paragrafı tek bir metinde topluyoruz
(ısınmadaki döngü):

In [ ]:
bilgiler = ""
for paragraf, skor in sirali.most_common(3):
    bilgiler = bilgiler + paragraf + "\n"

print(bilgiler)

Yönerge üç parçadan oluşacak: **talimat + bilgiler + soru**. Talimat modele iki şey söylüyor: yalnızca bu bilgilere dayan;
bilgilerde cevap yoksa bunu açıkça söyle.

In [ ]:
talimat = "Yalnızca aşağıdaki bilgilere dayanarak cevap ver. Bilgilerde cevap yoksa yalnızca 'Bilmiyorum' yaz."

Şablonu **sen yaz**: `yonerge` adında bir metin kur. Sırasıyla `talimat`, iki yeni satır (`"\n\n"`), `"Bilgiler:\n"`,
`bilgiler`, bir yeni satırla `"Soru: "` ve en sonda `soru`. Parçaları `+` ile birleştir, sonra yazdır: model tam olarak
bunu okuyacak.

Bir çözüm:

In [ ]:
yonerge = talimat + "\n\nBilgiler:\n" + bilgiler + "\nSoru: " + soru
print(yonerge)

**Tahmin et:** model bu kez ne diyecek? Bir istek gider; iki cevabı alt alta yazdırıyoruz.

In [ ]:
kilavuzlu = modele_sor(yonerge, 0.2)

print("KILAVUZSUZ:", kilavuzsuz)
print()
print("KILAVUZLU:", kilavuzlu)

İki cevabı karşılaştır, senin çıktına bak:

- Kılavuzlu cevapta mürekkep mavisi ve kâğıt beyazı var mı? Renk kodları kılavuzdakilerle aynı mı?
- Kırmızı–turuncu yasağından söz ediyor mu? Etmiyorsa Adım 4'e dön: o paragraf ilk üçe girmiş miydi?
- Kılavuzlu cevap, bilgilerde olmayan bir şey ekledi mi?

Model yalnızca yönergede gördüğünü bilir; kendisine gelmeyen bir paragrafı bilemez. RAG'de cevap, **bulma** adımından
daha iyi olamaz.

## Adım 6 — Fonksiyona koy: `kilavuzda_ara`

Adım 4 ve 5'te yazdıklarını iki fonksiyona koyuyoruz; böylece yeni bir soru sormak tek satır olur.

`kilavuzda_ara(soru)`: soruyu alır, kılavuzun en ilgili üç paragrafını **tek bir metin** olarak verir.
Konu 12'de bu fonksiyon ajanın araçlarından biri olacak; adı ve işi aynı kalacak.

In [ ]:
def kilavuzda_ara(soru):
    soru_vektoru = vektor_al(soru)
    skorlar = {}
    for paragraf in kilavuz_vektorleri:
        vektor = kilavuz_vektorleri[paragraf]
        skorlar[paragraf] = benzerlik(soru_vektoru, vektor)
    sirali = Counter(skorlar)
    bilgiler = ""
    for paragraf, skor in sirali.most_common(3):
        bilgiler = bilgiler + paragraf + "\n"
    return bilgiler

`kilavuza_sor(soru)`: kılavuzda arar, senin şablonunla yönergeyi kurar ve modele sorar.

In [ ]:
def kilavuza_sor(soru):
    bilgiler = kilavuzda_ara(soru)
    yonerge = talimat + "\n\nBilgiler:\n" + bilgiler + "\nSoru: " + soru
    return modele_sor(yonerge, 0.2)

Şimdi kılavuzda **olmayan** bir soru. Kılavuz, menü kartında fiyatların nereye yazılacağını söylüyor; fiyatların kendisini
söylemiyor.

**Tahmin et, sohbete yaz:**

1. `kilavuzda_ara` bu soruya hiç paragraf vermeyecek mi?
2. Model "Bilmiyorum" mu diyecek, yoksa bir fiyat mı yazacak?

In [ ]:
fiyat_sorusu = "Sayfa Kahve'de bir fincan filtre kahve kaç lira?"

bulunan = kilavuzda_ara(fiyat_sorusu)
print(bulunan)

Birinci sorunun cevabı: her zaman üç paragraf gelir. "En yakın" her zaman vardır, ilgili olmasa da.
Gelen paragraflarda "fiyat" sözcüğü geçen var mı? Fiyatın kendisi var mı?

In [ ]:
fiyat_cevabi = kilavuza_sor(fiyat_sorusu)
print(fiyat_cevabi)

Senin çıktına bak: model "Bilmiyorum" dedi mi? Bir fiyat yazdıysa o fiyat, verdiğimiz bilgilerin hiçbirinde yok; talimata
rağmen uydurdu. Talimat modeli yönlendirir ama zorlamaz (Konu 5: model talimatı kesin bir kural olarak değil, olası bir
devam olarak okur). RAG uydurmayı azaltır, sıfırlamaz. Cevaptaki her bilgiyi kılavuzla karşılaştırmak yine senin işin.

## Adım 7 — Kılavuza uygun bir afiş

Şimdi tasarım işi: kılavuza uygun bir afiş görseli. Görsel yönergesini metin modeline yazdıracağız; ama önce kılavuzun
**görselle ilgili** kurallarını bulup ona vereceğiz.

Arama sorusunu dikkatle seçiyoruz: afişin konusunu (sınav haftası) değil, görselin kurallarını arıyoruz.

In [ ]:
brif = "Sınav haftasında kafenin gece yarısına kadar açık olduğunu duyuran bir afiş"
arama = "Afiş görselinde hangi renkler ve hangi çizim tarzı kullanılır, neler gösterilmez?"

kurallar = kilavuzda_ara(arama)
print(kurallar)

Gelen üç paragrafa bak: renk, çizim tarzı ve insan yüzü kurallarının üçü de geldi mi? Hangisi eksik?

Görsel model İngilizce yönergeyle daha iyi çalışır (Konu 7). Metin modelinden kurallara uyan, İngilizce bir görsel yönergesi
istiyoruz. Görselde yazı istemiyoruz: harfleri görsel model bozar, başlığı tasarımcı dizer (Konu 10). Sıcaklık 0.7:
burada biraz yaratıcılık istiyoruz.

In [ ]:
istek = "Şu kurallara uyan bir afiş görseli için İngilizce, tek paragraflık bir görsel yönergesi yaz. "
istek = istek + "Görselde yazı olmasın. Yalnızca yönergeyi yaz.\n\nKurallar:\n" + kurallar
istek = istek + "\nAfiş: " + brif

gorsel_yonergesi = modele_sor(istek, 0.7)
print(gorsel_yonergesi)

**Tahmin et:** bu yönergeyle çizilen afişte kılavuzun hangi kuralı tutar, hangisi tutmaz?

`ciz` görseli verdiğin yola kaydeder; bir görsel isteği gider (birkaç saniye sürer).

In [ ]:
yol = ciz(gorsel_yonergesi, 42, "ciktilar/afis_kilavuzlu.jpg")
print(yol)

`ciktilar/afis_kilavuzlu.jpg`'yi aç ve kılavuzla karşılaştır. Sohbete yaz:

- Renkler kılavuza uyuyor mu? Kırmızı ya da turuncu var mı?
- İnsan yüzü var mı?
- Çizim tarzı risografa benziyor mu?

Bir kural iki yerde kaybolabilir: **aramada** (kural modele hiç gitmedi) ya da **çizimde** (kural yönergede vardı ama görsel
model uygulamadı). Uymayan bir kural gördüysen önce `gorsel_yonergesi`'ne bak: kural orada var mı? Hangisi olduğunu
bulursan neyi düzelteceğini de bulursun: arama sorusunu mu, görsel yönergesini mi?

## Bonus

**1. Kılavuzsuz afiş.** Aynı brif, ama kılavuz yok. Bir metin, bir görsel isteği gider. İki afişi yan yana aç:
hangisi Sayfa Kahve'ye ait?

In [ ]:
istek = "Sayfa Kahve adlı bir kafe için şu afişin görselini anlatan İngilizce, tek paragraflık bir görsel yönergesi yaz. "
istek = istek + "Görselde yazı olmasın. Yalnızca yönergeyi yaz.\n\nAfiş: " + brif

kilavuzsuz_yonerge = modele_sor(istek, 0.7)
print(kilavuzsuz_yonerge)
yol = ciz(kilavuzsuz_yonerge, 42, "ciktilar/afis_kilavuzsuz.jpg")

**2. Kaç paragraf?** Adım 6'daki `kilavuzda_ara` hücresinde `most_common(3)`'ü `most_common(1)` yap ve o hücreyi yeniden
çalıştır. Sonra aşağıdaki hücreyi çalıştır: Adım 1'deki renk sorusu, tek paragrafla. Cevap Adım 5'tekine göre ne kadar eksik?

Sonra `most_common(14)` dene: her soruda kılavuzun tamamı gider. On dört paragraflık bir kılavuz için bu olur;
iki yüz sayfalık bir kurumsal kimlik kılavuzu için neden olmaz? Bitince 3'e geri döndür.

In [ ]:
renk_cevabi = kilavuza_sor(soru)
print(renk_cevabi)

**3. Yarım bilgi.** Kılavuz başlık yazı tipini tarif ediyor (serifli, eski kitap baskısını andıran) ama adını vermiyor.
Model bir ad uyduracak mı?

In [ ]:
yazi_sorusu = "Sayfa Kahve'nin başlıklarında hangi yazı tipini kullanmalıyım? Yazı tipinin adını yaz."

yazi_cevabi = kilavuza_sor(yazi_sorusu)
print(yazi_cevabi)

Bir yazı tipi adı yazdıysa o ad kılavuzda yok. Tasarımcı için iyi bir öneri olabilir; ama müşteriye "kılavuz böyle diyor"
diye götürülemez.

## Sıradaki konu

Bugün kılavuza ne zaman bakılacağına sen karar verdin: her soruda önce `kilavuzda_ara`, sonra model. Konu 12'de
`kilavuzda_ara` bir **araç** olacak. Ajana "kafeye kılavuza uygun bir afiş yap" dediğinde önce kılavuza bakmaya, sonra
çizmeye model kendisi karar verecek.